# DOME Flow Matching + Resample 训练流程

这个 notebook 按脚本顺序带你跑完整训练流程：

1. 检查数据、checkpoint 和依赖
2. 准备 resample 数据
3. 启动只训练 DOME 的 flow matching 训练
4. 打开 TensorBoard 看指标
5. 评估 checkpoint
6. 可视化预测结果

注意：OccVAE 在这个流程里只加载 checkpoint 并冻结，不会训练。真正更新参数的是 DOME / world model。

## 0. 进入项目根目录

在服务器上打开这个 notebook 时，先确认当前目录是 `dome-cfm` 项目根目录。下面这个 cell 会自动检查关键文件。

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

ROOT = Path.cwd()

# 如果 notebook 不是从项目根目录启动，可以手动改这里：
# ROOT = Path('/mnt/data2/whz/dome-cfm')

os.chdir(ROOT)
print('当前目录:', ROOT)
print('训练脚本存在:', (ROOT / 'tools/train_dome_flow_resample.sh').exists())
print('配置文件存在:', (ROOT / 'config/train_dome_flow_resample.py').exists())

## 1. 配置本次实验路径

下面这些变量会传给 shell 脚本。默认路径和我给你写的脚本保持一致。

如果你的数据或 checkpoint 放在别的地方，只改下面这个 cell 就行。

In [ ]:
# 基础配置
CFG = './config/train_dome_flow_resample.py'
WORK_DIR = './work_dir/dome_flow_resample'
VAE_CKPT = 'ckpts/occvae_latest.pth'

# resample 数据准备相关
DATA_PATH = './data/nuscenes'
IMAGESET = './data/nuscenes_infos_train_temporal_v3_scene.pkl'
RESAMPLE_DST = './data/resampled_occ'

# 训练相关
CUDA_VISIBLE_DEVICES = '0'
SEED = '42'
TB_DIR = './work_dir/dome_flow_resample/tb_log'

# 如果从已有 DOME 权重微调，填这个；从头训练就保持空字符串。
LOAD_FROM = ''

# 如果从中断 checkpoint 继续训练，填这个；不继续就保持空字符串。
RESUME_FROM = ''

# 评估/可视化相关
DOME_CKPT = './work_dir/dome_flow_resample/latest.pth'
SCENE_IDX = '6 7'
NUM_SAMPLING_STEPS = '20'

print('CFG:', CFG)
print('WORK_DIR:', WORK_DIR)
print('VAE_CKPT:', VAE_CKPT)
print('RESAMPLE_DST:', RESAMPLE_DST)

## 2. 定义运行命令的小工具

`run_cmd()` 会打印命令，然后执行。长时间任务，比如训练和 resample，会直接把日志输出在 notebook 里。

如果你只是想先看命令，把 `dry_run=True`。

In [ ]:
def run_cmd(cmd, env=None, dry_run=False):
    env_all = os.environ.copy()
    if env:
        env_all.update({k: str(v) for k, v in env.items() if v is not None})
    print('\n命令:')
    print(' '.join(shlex.quote(str(x)) for x in cmd))
    if env:
        print('\n环境变量:')
        for k, v in env.items():
            if v not in (None, ''):
                print(f'{k}={v}')
    if dry_run:
        print('\n这是 dry run，没有真正执行。')
        return None
    return subprocess.run(cmd, env=env_all, check=True)

## 3. 检查训练前置条件

这一步等价于运行：

```bash
python tools/check_dome_resample_setup.py
```

它会检查配置、OccVAE checkpoint、nuScenes 数据、resample 数据和 `torchcfm`。如果只缺 `data/resampled_occ`，继续下一步生成即可。

In [ ]:
run_cmd([
    'python', 'tools/check_dome_resample_setup.py',
    '--py-config', CFG,
    '--vae-ckpt', VAE_CKPT,
    '--resample-root', RESAMPLE_DST,
    '--data-root', DATA_PATH,
    '--train-imageset', IMAGESET,
])

## 4. 生成 resample 数据

如果上一步提示缺少 `data/resampled_occ`，运行这个 cell。

这一步会比较久，因为它会根据 train pkl 逐个 scene 生成 resample occupancy。已经生成过的话可以跳过。

In [ ]:
RUN_RESAMPLE = False  # 需要生成 resample 数据时改成 True

if RUN_RESAMPLE:
    run_cmd(
        ['bash', 'tools/prepare_resample_data.sh'],
        env={
            'DATA_PATH': DATA_PATH,
            'IMAGESET': IMAGESET,
            'DST': RESAMPLE_DST,
        }
    )
else:
    print('已跳过 resample 数据生成。需要运行时，把 RUN_RESAMPLE 改成 True。')

## 5. 再检查一次

生成 resample 后建议再检查一次。全部 `[OK]` 后再开始训练。

In [ ]:
run_cmd([
    'python', 'tools/check_dome_resample_setup.py',
    '--py-config', CFG,
    '--vae-ckpt', VAE_CKPT,
    '--resample-root', RESAMPLE_DST,
    '--data-root', DATA_PATH,
    '--train-imageset', IMAGESET,
])

## 6. 启动训练

这一步会运行：

```bash
bash tools/train_dome_flow_resample.sh
```

脚本内部会调用 `tools/train_diffusion.py`，但配置里 `sample_method='flow'`，所以实际走的是 flow matching。

关键点：OccVAE 会从 `VAE_CKPT` 加载，并在训练脚本中冻结；DOME / world model 才会更新参数。

In [ ]:
RUN_TRAIN = False  # 真正开始训练时改成 True

train_env = {
    'CUDA_VISIBLE_DEVICES': CUDA_VISIBLE_DEVICES,
    'CFG': CFG,
    'WORK_DIR': WORK_DIR,
    'VAE_CKPT': VAE_CKPT,
    'LOAD_FROM': LOAD_FROM,
    'RESUME_FROM': RESUME_FROM,
    'TB_DIR': TB_DIR,
    'SEED': SEED,
}

if RUN_TRAIN:
    run_cmd(['bash', 'tools/train_dome_flow_resample.sh'], env=train_env)
else:
    run_cmd(['bash', 'tools/train_dome_flow_resample.sh'], env=train_env, dry_run=True)
    print('\n确认路径没问题后，把 RUN_TRAIN 改成 True 再运行这个 cell。')

## 7. TensorBoard 看训练指标

训练脚本默认会把 TensorBoard 日志写到：

```text
WORK_DIR/tb_log
```

主要看这些指标：

- `train/loss`
- `train/mse`
- `train/lr`
- `train/grad_norm`
- `val/iou`
- `val/miou`

在服务器上通常用 `--host 0.0.0.0`，然后通过端口转发或服务器面板打开。

In [ ]:
print('TensorBoard 日志目录:', TB_DIR)
print('启动命令:')
print(f'tensorboard --logdir {shlex.quote(TB_DIR)} --host 0.0.0.0 --port 6006')

## 8. 评估 checkpoint

训练完成或保存了某个 epoch 后，运行评估脚本。

如果 `DOME_CKPT` 不存在，脚本会自动尝试从 `WORK_DIR` 里找最新的 `epoch_*.pth`。

In [ ]:
RUN_EVAL = False  # 要评估时改成 True

eval_env = {
    'CUDA_VISIBLE_DEVICES': CUDA_VISIBLE_DEVICES,
    'CFG': CFG,
    'WORK_DIR': WORK_DIR,
    'VAE_CKPT': VAE_CKPT,
    'DOME_CKPT': DOME_CKPT,
    'SEED': SEED,
}

if RUN_EVAL:
    run_cmd(['bash', 'tools/eval_dome_flow_resample.sh'], env=eval_env)
else:
    run_cmd(['bash', 'tools/eval_dome_flow_resample.sh'], env=eval_env, dry_run=True)
    print('\n需要评估时，把 RUN_EVAL 改成 True。')

## 9. 可视化预测结果

这一步会把预测 occupancy 画出来，并生成视频。

默认看 scene 6 和 scene 7。你可以改 `SCENE_IDX`，比如：

```python
SCENE_IDX = '6 7 16 18'
```

`NUM_SAMPLING_STEPS` 是 flow matching 采样步数，越大通常越慢。

In [ ]:
RUN_VIS = False  # 要可视化时改成 True

vis_env = {
    'CUDA_VISIBLE_DEVICES': CUDA_VISIBLE_DEVICES,
    'CFG': CFG,
    'WORK_DIR': WORK_DIR,
    'VAE_CKPT': VAE_CKPT,
    'DOME_CKPT': DOME_CKPT,
    'SCENE_IDX': SCENE_IDX,
    'NUM_SAMPLING_STEPS': NUM_SAMPLING_STEPS,
}

if RUN_VIS:
    run_cmd(['bash', 'tools/vis_dome_flow_resample.sh'], env=vis_env)
else:
    run_cmd(['bash', 'tools/vis_dome_flow_resample.sh'], env=vis_env, dry_run=True)
    print('\n需要可视化时，把 RUN_VIS 改成 True。')

## 10. 训练过程中到底发生了什么

一次训练迭代可以理解成：

```text
1. 从 data/resampled_occ 里取 11 帧 occupancy
2. OccVAE encoder 把 occupancy 压缩成 latent
3. 随机造一份噪声 x0
4. 真实 latent 当作 x1
5. flow matching 在 x0 和 x1 中间采样 x_t
6. DOME 根据 x_t、时间 t、前 4 帧条件和 pose 预测速度方向
7. 用 MSE 比较预测速度和真实速度
8. 只更新 DOME 参数
9. OccVAE 不更新
```

所以这次训练不是重新训练整个系统，而是：

```text
冻结 OccVAE，只训练 DOME，让 DOME 学会用 flow matching 生成未来 occupancy latent。
```

## 11. 最常用命令汇总

如果不想用 notebook，也可以直接在终端跑：

```bash
cd /mnt/data2/whz/dome-cfm
conda activate torchcfm

python tools/check_dome_resample_setup.py
bash tools/prepare_resample_data.sh
CUDA_VISIBLE_DEVICES=0 bash tools/train_dome_flow_resample.sh
tensorboard --logdir ./work_dir/dome_flow_resample/tb_log --host 0.0.0.0 --port 6006
bash tools/eval_dome_flow_resample.sh
SCENE_IDX="6 7" bash tools/vis_dome_flow_resample.sh
```